## IRFL dataset - testing

In [1]:
import sys
import os

sys.path.append(os.getcwd())
# append the root directory to the sys.path

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

%load_ext autoreload
%autoreload 2

### Download some necessary packages

In [ ]:
!pip install transformers --quiet
!pip install -U datasets --quiet
!pip install pip install tqdm --quiet
!pip install fsspec==2023.6.0 --quiet

### Load data from HF

In [88]:
from datasets import Dataset
Dataset.cleanup_cache_files
from datasets import load_dataset
import pandas as pd

IRFL_images = load_dataset("lampent/IRFL", data_files='IRFL_images.zip')['train']

# IRFL dataset of figurative phrase-image pairs (10k+ images)
IRFL_idioms_dataset = load_dataset("lampent/IRFL", 'idioms-dataset')['dataset']
IRFL_similes_dataset = load_dataset("lampent/IRFL", 'similes-dataset')['dataset']
IRFL_metaphors_dataset = load_dataset("lampent/IRFL", 'metaphors-dataset')['dataset']

print('Successfully loaded IRFL dataset and tasks')

Successfully loaded IRFL dataset and tasks


In [89]:
IRFL_similes_dataset

Dataset({
    features: ['phrase', 'property', 'source', 'figurative_type', 'uuid', 'concept', 'concept_type', 'category'],
    num_rows: 2923
})

### Convert data for easier manipulation

In [90]:
# covert datasets to pandas dataframes for easier manipulation
IRFL_idioms_df = pd.DataFrame(IRFL_idioms_dataset)
IRFL_similes_df = pd.DataFrame(IRFL_similes_dataset)
IRFL_metaphors_df = pd.DataFrame(IRFL_metaphors_dataset)

IRFL_idioms_csv = IRFL_idioms_df.to_csv('IRFL_idioms_dataset.csv', index=False)
IRFL_similes_csv = IRFL_similes_df.to_csv('IRFL_similes_dataset.csv', index=False)
IRFL_metaphors_csv = IRFL_metaphors_df.to_csv('IRFL_metaphors_dataset.csv', index=False)

In [91]:
IRFL_idioms_df.head()

,IRFL_id,annotations,category,literal_candidate,phrase,query,rank,source,uuid,definition,figurative_type
0,6372933026274098672734891785206981608478080193...,"['None', 'None', 'None', 'None', 'None']",None,False,a little bird told me,I received the information from a source not t...,1,https://www.science.org/doi/10.1126/sciadv.aay...,6372933026274098672734891785206981608478080193...,['I received the information from a source not...,idiom
1,2264012059128528847681353831285332760714914694...,"['Figurative+Literal', 'Literal', 'Figurative+...",Figurative+Literal,True,a little bird told me,a little bird told me,2,https://www.vectorstock.com/royalty-free-vecto...,2264012059128528847681353831285332760714914694...,['I received the information from a source not...,idiom
2,2970500809659499579266821173012282594209082880...,"['Literal', 'Literal', 'Literal', 'Partial Lit...",Literal,True,a little bird told me,a little bird told me,1,https://www.istockphoto.com/illustrations/a-li...,2970500809659499579266821173012282594209082880...,['I received the information from a source not...,idiom
3,1066704153386479799672335240437076214661283009...,"['Figurative+Literal', 'Literal', 'Figurative+...",Figurative+Literal,True,a little bird told me,a little bird told me,3,https://joryfisher.com/2014/09/a-little-birdie...,1066704153386479799672335240437076214661283009...,['I received the information from a source not...,idiom
4,1515299820314437205492113156169356771249304209...,"['Figurative+Literal', 'Literal', 'Partial Lit...",No Category,True,a little bird told me,a little bird told me,4,https://www.function1.com/2012/06/a-little-bir...,1515299820314437205492113156169356771249304209...,['I received the information from a source not...,idiom


In [ ]:
from datasets import load_dataset
from pathlib import Path
from urllib.parse import urlparse, urljoin
import hashlib, json, re, time
import requests
from bs4 import BeautifulSoup

DATASET="lampent/IRFL"
CONFIG="idiom-detection-task"
SPLIT="test"

OUT = Path("irfl_images")/CONFIG/SPLIT
OUT.mkdir(parents=True, exist_ok=True)
MANIFEST = OUT/"manifest.jsonl"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

# domains that very often block non-browser requests from server IPs
HARD_BLOCK = {"science.org", "nature.com", "cell.com", "sciencedirect.com"}

def safe(s): 
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", str(s))[:120]

def is_image_response(resp):
    return ("image" in (resp.headers.get("Content-Type","").lower()))

def og_image_from_html(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    for key in [("property","og:image"), ("name","twitter:image")]:
        tag = soup.find("meta", attrs={key[0]: key[1]})
        if tag and tag.get("content"):
            return urljoin(base_url, tag["content"])
    return None

def fetch_bytes(url, headers=None, timeout=30):
    return requests.get(url, headers=headers or HEADERS, timeout=timeout, allow_redirects=True)

def resolve_image_url(source_url):
    host = urlparse(source_url).netloc.lower()
    if any(host.endswith(d) for d in HARD_BLOCK):
        return None, "hard_block_domain"

    r = fetch_bytes(source_url)
    if r.status_code in (401,403):
        return None, f"blocked_{r.status_code}"

    if r.ok and is_image_response(r):
        return source_url, "direct"

    # if html is accessible, try OG image
    if r.ok:
        og = og_image_from_html(r.text, source_url)
        if og:
            r2 = fetch_bytes(og)
            if r2.ok and is_image_response(r2):
                return og, "og:image_ok"
            return None, "og:image_not_image_or_blocked"

    return None, f"unhandled_status_{r.status_code}"

ds = load_dataset(DATASET, CONFIG, split=SPLIT)

seen = set()
ok = fail = 0

with MANIFEST.open("w", encoding="utf-8") as mf:
    for row_idx, row in enumerate(ds):
        meta = row.get("images_metadata")
        
        if not isinstance(meta, list):
            
            continue

        for m in meta:
            source = m.get("source")
            if not source or source in seen:
                print("  skipping source:", source)
                continue
            seen.add(source)

            img_url, how = resolve_image_url(source)
            record = {
                "row_idx": row_idx,
                "source": source,
                "resolved_image_url": img_url,
                "resolution": how,
                "uuid": m.get("uuid"),
                "IRFL_id": m.get("IRFL_id"),
                "category": m.get("category"),
                "phrase": row.get("phrase"),
                "query": row.get("query"),
            }

            if img_url:
                # filename based on hash of resolved URL
                h = hashlib.md5(img_url.encode("utf-8")).hexdigest()[:12]
                path = OUT/f"{h}.jpg"

                if not path.exists():
                    r = fetch_bytes(img_url)
                    if r.ok and is_image_response(r):
                        path.write_bytes(r.content)
                    else:
                        record["resolved_image_url"] = None
                        record["resolution"] = "resolved_but_download_failed"
                        fail += 1
                        mf.write(json.dumps(record, ensure_ascii=False)+"\n")
                        continue

                record["file"] = str(path)
                ok += 1
            else:
                fail += 1

            mf.write(json.dumps(record, ensure_ascii=False)+"\n")

            time.sleep(0.15)

print("downloaded:", ok, "failed:", fail, "manifest:", MANIFEST)



AttributeError: 'str' object has no attribute 'to_list'

In [ ]:
from tqdm import tqdm
from urllib.request import urlopen


def process_df(df):
  distractors_ids = df['distractors'].to_list()
  answers_ids = df['distractors'].to_list()
  phrases = df['phrase'].to_list()
  fig_types = df['figurative_type'].to_list()

  distractors = []
  answers = []
  texts = []
  types = []

  for i in tqdm(range(len(distractors_urls)), total=len(distractors_urls)):

    d_urls = distractors_urls[i]
    distractor = [Image.open(urlopen(url)) for url in eval(d_urls)]

    a_urls = answers_urls[i]
    answer = Image.open(urlopen(eval(a_urls)[0]))

    text = phrases[i]
    fig_type = fig_types[i]

    distractors.append(distractor)
    answers.append(answer)
    texts.append(text)
    types.append(fig_type)
      

  return distractors, answers, texts, types



def collate_fn(batch):
    images = [data[0] for data in batch]
    texts = [data[1] for data in batch]
    labels = [data[2] for data in batch]

    return images, texts, torch.tensor(labels, dtype=int)


class FigTypeDataset(Dataset):
    def __init__(self, answers, texts, types):
        self.types= types
        self.images = answers
        self.texts = texts

        self.type_map = {'idiom': 0, 'simile': 1, 'metaphor': 2}

        self.labels = list(map(lambda x: self.type_map[x], self.types))


    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.texts[idx], self.labels[idx]

In [47]:
def process_fn(batch):
    images, texts, contrastive_labels = batch
    batch = processor(images=images, text=texts, padding=True, return_tensors='pt')

    return batch, contrastive_labels

In [69]:
distractors_simile, answers_simile, texts_simile, types_simile = process_df(IRFL_similes_df)
distractors_idiom, answers_idiom, texts_idiom, types_idiom = process_df(IRFL_idioms_df)
distractors_metaphor, answers_metaphor, texts_metaphor, types_metaphor = process_df(IRFL_metaphors_df)

  0%|          | 0/277 [00:00<?, ?it/s]


ValueError: unknown url type: '40775837622118919445361671246634665833737407213773351532367458923460437463700'